In [1]:
# pip install mygene

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

In [2]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [3]:
df.head()

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN


In [71]:
# df_genes = df["gene_symbol"]

In [72]:
# df_genes

In [73]:
# df_genes_t = df_genes.dropna()

In [74]:
# df_genes_t = df_genes_t.reset_index(drop=False)["gene_symbol"]

In [75]:
# df_genes_t

In [76]:
# df_genes_t.head(20)

In [4]:
def extrae_gene_symbols(dataframe):
    """
    Función destinada a limpiar, acondicionar y extraer la columna 'gene_symbol' a partir del DataFrame original. Elimina valores <na> de
    dicha columna, elimina posibles gene symbols que incluyan el término 'dist' que complica las búsquedas posteriores, y separa en líneas
    diferentes los gene symbols que incluyen varios genes separados por ',' o ';'.

    Parámetros
    -------------------------------
    - dataframe (pandas.core.frame.DataFrame) -> DataFrame original importado desde t_common_variant.txt

    Returns
    -------------------------------
    - df_corregido (pandas.core.frame.DataFrame) -> DataFrame original corregido tras eliminar columna espuria al final, y valores <na> 
                                                    de la columna 'gene_symbol'
    - lista_symbols (list) -> lista con todos los gene symbols de la columna 'gene_symbols', tras separar los genes que aparecen 
                              originalmente unidos por ',' o ';', y eliminar las entradas con 'dist' y valores <na>
    - lista_unicos (list) -> lista todos los valores diferentes presentes en lista_symbols, sin repetidos.
    """

    df_corregido = df.drop('Unnamed: 10', axis = 1)
    df_corregido = df_corregido.dropna(subset = ['gene_symbol'])
    df_corregido = df_corregido.reset_index(drop=False)
    df_corregido = df_corregido.drop('index', axis = 1)
    df_genes = df_corregido["gene_symbol"]
    
    lista_symbols = []
    
    for i, gene in enumerate(df_genes):
        
        gene = gene.replace(",", ";")
    
        if "dist" in gene:
            continue

        elif ";" in gene:

            separacion1 = gene.split(";")

            for gen in separacion1:
                lista_symbols.append(gen)

        else:
            lista_symbols.append(gene)

    lista_unicos = []
    
    for symbol in lista_symbols:
        
        if symbol not in lista_unicos:
            
            lista_unicos.append(symbol)
                
    return df_corregido, lista_symbols, lista_unicos

In [37]:
df_corregido, lista_symbols, lista_unicos = extrae_gene_symbols(df)

In [38]:
def extrae_SNPs(df_corregido):
    """
    Función destinada a adaptar df_corregido para que contenga únicamente las columnas de interés para el manejo de los SNPs ('Chr', 
    'gene_symbol', 'SNP_position', 'effect_allele', 'alternate_allele'). Se separan en líneas diferentes las entradas correspondientes 
    a los genes que se presentan unidos por ',' o ';' en df_corregido. Se eliminan las entradas espurias que, por motivos ajenos, ya sea
    por el funcionamiento del procesador de texto o el estado del dataset original, presentan datos corruptos, como el SNP_symbol en la 
    columna de cromosomas 'Chr'.

    Parámetros
    -------------------------------
    - df_corregido (pandas.core.frame.DataFrame) -> DataFrame original corregido tras eliminar columna espuria al final, y valores <na> 
                                                    de la columna 'gene_symbol'
    Returns
    -------------------------------
    - df_SNPs (pandas.core.frame.DataFrame) -> DataFrame con las columnas ('Chr', 'gene_symbol', 'SNP_position', 'effect_allele', 
    'alternate_allele') limpias y acondicionadas.
    """
    df_corregido["gene_symbol"] = df_corregido["gene_symbol"].str.split(r'\s*[;,]\s*')

    df_corregido = df_corregido.explode("gene_symbol").reset_index(drop = True)

    df_SNPs = df_corregido[["Chr", "gene_symbol", "SNP_position", "effect_allele", "alternate_allele"]]

    df_SNPs = df_SNPs[~df_SNPs['Chr'].str.contains('rs', na = False)]

    df_SNPs = df_SNPs[~df_SNPs['gene_symbol'].str.contains('dist')].reset_index(drop = True)

    df_SNPs = df_SNPs[df_SNPs['gene_symbol'] != 'NONE'].reset_index(drop = True)

    df_SNPs = df_SNPs[df_SNPs['Chr'] != '-'].reset_index(drop = True)
    
    return df_SNPs

In [39]:
df_SNPs = extrae_SNPs(df_corregido)

In [40]:
df_SNPs

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele
0,6,GPR126,142758601,T,G
1,1,SYT11,155839054,C,T
2,12,SLC2A13,40428561,G,T
3,12,SLC2A13,40478652,G,T
4,12,SLC2A13,40474147,C,T
...,...,...,...,...,...
1344,17,DNAH17,76425480,A,T
1345,18,ASXL3,31304318,T,G
1346,18,MEX3C,48683589,T,G
1347,20,CRLS1,6006041,T,C


In [15]:
lista_genes_snps = []
lista_unicos_snps = []

for i in range(len(df_SNPs)):
    if df_SNPs.iloc[i]["gene_symbol"] not in lista_genes_snps:
        lista_genes_snps.append(df_SNPs.iloc[i]["gene_symbol"])
        lista_unicos_snps.append(df_SNPs.iloc[i]["gene_symbol"])
    else:
        lista_genes_snps.append(df_SNPs.iloc[i]["gene_symbol"])        

In [16]:
diccionario = {i: [] for i in lista_unicos_snps}

In [17]:
for i in range(len(df_SNPs)):
    if len(diccionario[df_SNPs.iloc[i]['gene_symbol']]) < 1:
        diccionario[df_SNPs.iloc[i]['gene_symbol']] = [df_SNPs.iloc[i]['Chr']]
    
    elif len(diccionario[df_SNPs.iloc[i]['gene_symbol']]) >= 1:
        diccionario[df_SNPs.iloc[i]['gene_symbol']].append(df_SNPs.iloc[i]['Chr'])

    else:
        continue

In [19]:
# diccionario

In [42]:
conteo_chr = df_SNPs.groupby("gene_symbol")["Chr"].nunique()

In [43]:
genes_problema = conteo_chr[conteo_chr > 1].index

In [44]:
df_SNPs = df_SNPs[~df_SNPs["gene_symbol"].isin(genes_problema)]

In [45]:
df_SNPs = df_SNPs.reset_index(drop = True)

In [46]:
df_SNPs

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele
0,6,GPR126,142758601,T,G
1,1,SYT11,155839054,C,T
2,12,SLC2A13,40428561,G,T
3,12,SLC2A13,40478652,G,T
4,12,SLC2A13,40474147,C,T
...,...,...,...,...,...
1282,17,DNAH17,76425480,A,T
1283,18,ASXL3,31304318,T,G
1284,18,MEX3C,48683589,T,G
1285,20,CRLS1,6006041,T,C


In [19]:
diccionario = {i: [] for i in lista_unicos}

In [20]:
for i in range(len(df_SNPs)):
    if len(diccionario[df_SNPs.iloc[i]['gene_symbol']]) < 1:
        diccionario[df_SNPs.iloc[i]['gene_symbol']] = [df_SNPs.iloc[i]['Chr']]
    
    elif len(diccionario[df_SNPs.iloc[i]['gene_symbol']]) >= 1:
        diccionario[df_SNPs.iloc[i]['gene_symbol']].append(df_SNPs.iloc[i]['Chr'])

    else:
        continue

In [22]:
# diccionario

In [47]:
chr_esperado = dict(zip(df_SNPs["gene_symbol"], df_SNPs["Chr"].astype(str)))

In [48]:
chr_esperado

{'GPR126': '6',
 'SYT11': '1',
 'SLC2A13': '12',
 'SLC2A15': '14',
 'LINC02471': '12',
 'LRRK2': '12',
 'GPRIN3': '4',
 'SNCA': '4',
 'LINC02210-CRHR1': '17',
 'SERPINA1': '14',
 'S1PR1': '1',
 'OLFM3': '1',
 'SPPL2C': '17',
 'HLA-DRA': '6',
 'DGKQ': '4',
 'NSF': '17',
 'PRRG4': '11',
 'QSER1': '11',
 'GAK': '4',
 'WNT3': '17',
 'CSMD1': '8',
 'TAS2R19': '12',
 'UNC13B': '9',
 'LINC00693': '3',
 'MPHOSPH10': '2',
 'ZNF519': '18',
 'AAK1': '2',
 'DHRS2': '14',
 'DHRS4-AS1': '14',
 'C12orf75': '12',
 'CASC18': '12',
 'CDH6': '5',
 'GRB10': '7',
 'PIK3CD': '1',
 'LY75-CD302': '2',
 'PLA2R1': '2',
 'SH3GL2': '9',
 'MED13': '17',
 'TBC1D3P2': '17',
 'CAST': '5',
 'SLCO3A1': '15',
 'DNAH11': '7',
 'PLEKHM1': '17',
 'LINC01271': '20',
 'PTPN1': '20',
 'PCAT5': '10',
 'ANKRD30A': '10',
 'GABRB3': '15',
 'RNF130': '5',
 'FSCB': '14',
 'MCTP2': '15',
 'LOC440311': '15',
 'DPY19L3': '19',
 'PDCD5': '19',
 'C20orf78': '20',
 'SLC24A3': '20',
 'HLA-DQB1': '6',
 'HLA-DQA2': '6',
 'SLC22A3': '6',
 'S

In [25]:
# chr_esperado

In [71]:
def extrae_coords_inicio_fin_sinLOC(df_SNPs, lista_unicos_snps):

    chr_esperado = dict(zip(df_SNPs["gene_symbol"], df_SNPs["Chr"].astype(str)))
    
    mg = mygene.MyGeneInfo()
    
    resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")
    
    coords_genes = []
    no_encontrados = []
    genes_procesados = set()
    
    for resultado in resultados_coords:
    
        gen = resultado.get("query")
        
        if gen in genes_procesados:
            continue
    
        if resultado.get("notfound"):
            if gen not in no_encontrados:
                no_encontrados.append(gen)
            continue
    
        posiciones = resultado.get("genomic_pos_hg19")
        
        if not posiciones:
            continue
    
        if not isinstance(posiciones, list):
            posiciones = [posiciones]
    
        chr_buscado = chr_esperado.get(gen)
        posicion_elegida = None
    
        for pos in posiciones:
            if str(pos.get("chr")) == chr_buscado:
                posicion_elegida = pos
                break
    
        if posicion_elegida:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})
    
            genes_procesados.add(gen)
    
        if gen in no_encontrados:
            no_encontrados.remove(gen)

    # recuperados = []
    # for gen in no_encontrados:
    #     if 'LOC' in gen:
    #         cambio = gen.replace('LOC', '')
    #         recuperados.append(cambio)

    # recuperados_coords = mg.querymany(recuperados, scopes = "entrezgene", fields = "symbol,genomic_pos_hg19", species = "human")

    # for resultado in recuperados_coords:
    
    #     gen = resultado.get("query")
        
    #     if gen in genes_procesados:
    #         continue
    
    #     if resultado.get("notfound"):
    #         if gen not in no_encontrados:
    #             no_encontrados.append(gen)
    #         continue
    
    #     posiciones = resultado.get("genomic_pos_hg19")
        
    #     if not posiciones:
    #         continue
    
    #     if not isinstance(posiciones, list):
    #         posiciones = [posiciones]
    
    #     chr_buscado = chr_esperado.get(gen)
    #     posicion_elegida = None
    
    #     for pos in posiciones:
    #         if str(pos.get("chr")) == chr_buscado:
    #             posicion_elegida = pos
    #             break
    
    #     if posicion_elegida:
    #         coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})
    
    #         genes_procesados.add(gen)
    
    #     if gen in no_encontrados:
    #         no_encontrados.remove(gen)
  
    df_coords = pd.DataFrame(coords_genes)

    return df_coords, no_encontrados

In [72]:
df_coords, no_encontrados = extrae_coords_inicio_fin(df_SNPs, lista_unicos_snps)

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
48 input query terms found no hit:	['SLC2A15', 'NONE', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 


In [73]:
df_coords

,gene_symbol,chr,inicio,fin,cadena
0,GPR126,6,142622991,142767403,1
1,SYT11,1,155829300,155854990,1
2,SLC2A13,12,40148823,40499891,-1
3,LRRK2,12,40590546,40763087,1
4,GPRIN3,4,90157537,90229161,-1
...,...,...,...,...,...
567,DNAH17,17,76419778,76573476,-1
568,ASXL3,18,31158579,31331156,1
569,MEX3C,18,48700920,48744674,-1
570,CRLS1,20,5986736,6020699,1


In [124]:
def extrae_coords_inicio_fin(df_SNPs, lista_unicos_snps):

    chr_esperado = dict(zip(df_SNPs["gene_symbol"], df_SNPs["Chr"].astype(str)))
    
    mg = mygene.MyGeneInfo()
    
    resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")
    
    coords_genes = []
    no_encontrados = []
    genes_procesados = set()
    
    for resultado in resultados_coords:

        gen = resultado.get("query")
        
        if resultado.get("notfound"):
            if gen not in no_encontrados:
                no_encontrados.append(gen)
            continue

    recuperados = [gen.replace('LOC','') for gen in no_encontrados if 'LOC' in gen]

    recuperados_coords_aux = mg.querymany(recuperados, scopes = "entrezgene", fields = "symbol,genomic_pos_hg19", species = "human")
    
    nuevos_symbols = [recuperados_coords_aux[i]["symbol"] for i, gen in enumerate(recuperados_coords_aux) if not gen.get("notfound")]

    recuperados_coords = mg.querymany(nuevos_symbols, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

    for resultado in resultados_coords:

        gen = resultado.get("query")
        
        if gen in genes_procesados:
            continue
    
        posiciones = resultado.get("genomic_pos_hg19")
        
        if not posiciones:
            continue
    
        if not isinstance(posiciones, list):
            posiciones = [posiciones]
    
        chr_buscado = chr_esperado.get(gen)
        posicion_elegida = None
    
        for pos in posiciones:
            if str(pos.get("chr")) == chr_buscado:
                posicion_elegida = pos
                break
    
        if posicion_elegida:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})
    
            genes_procesados.add(gen)
    
        if gen in no_encontrados:
            no_encontrados.remove(gen)

    for resultado in recuperados_coords:

        gen = "LOC" + resultado.get("_id")
        
        if gen in genes_procesados:
            continue
    
        posiciones = resultado.get("genomic_pos_hg19")
        
        if not posiciones:
            continue
    
        if not isinstance(posiciones, list):
            posiciones = [posiciones]
    
        chr_buscado = chr_esperado.get(gen)
        posicion_elegida = None
    
        for pos in posiciones:
            if str(pos.get("chr")) == chr_buscado:
                posicion_elegida = pos
                break
    
        if posicion_elegida:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})
    
            genes_procesados.add(gen)
    
        if gen in no_encontrados:
            no_encontrados.remove(gen)

    df_coords = pd.DataFrame(coords_genes)

    return df_coords, no_encontrados

In [125]:
df_coords2, no_encontrados2 = extrae_coords_inicio_fin(df_SNPs, lista_unicos_snps)

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
48 input query terms found no hit:	['SLC2A15', 'NONE', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 
7 input query terms found no hit:	['201175', '100129900', '100129831', '645177', '100130911', '729160', '100132423']
12 input query terms found dup hits:	[('BALR6', 2), ('COX6CP4', 2), ('CLIC4P1', 2), ('CRIM1-DT', 2), ('SELENOKP3', 2), ('RPL7L1P5', 2), (


In [126]:
df_coords2

,gene_symbol,chr,inicio,fin,cadena
0,GPR126,6,142622991,142767403,1
1,SYT11,1,155829300,155854990,1
2,SLC2A13,12,40148823,40499891,-1
3,LRRK2,12,40590546,40763087,1
4,GPRIN3,4,90157537,90229161,-1
...,...,...,...,...,...
578,LOC100129138,1,104615645,104619709,1
579,LOC101929066,8,17942377,17953903,1
580,LOC100507657,22,27706612,27713417,1
581,LOC284930,22,48027423,48251349,1


In [49]:
# mg = mygene.MyGeneInfo()
    
# resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

# coords_genes = []
# no_encontrados = []
# genes_procesados = set()

# for resultado in resultados_coords:

#     gen = resultado.get("query")

#     if gen in genes_procesados:
#         continue

#     if resultado.get("notfound"):
#         if gen not in no_encontrados:
#             no_encontrados.append(gen)
#         continue

#     posiciones = resultado.get("genomic_pos_hg19")

#     if not posiciones:
#         continue

#     if not isinstance(posiciones, list):
#         posiciones = [posiciones]

#     chr_buscado = chr_esperado.get(gen)
#     posicion_elegida = None

#     for pos in posiciones:
#         if str(pos.get("chr")) == chr_buscado:
#             posicion_elegida = pos
#             break

#     if posicion_elegida:
#         coords_genes.append({"gene_symbol": gen, "chr": str(posicion_elegida.get("chr")), "inicio": posicion_elegida.get("start"), "fin": posicion_elegida.get("end"), "cadena": posicion_elegida.get("strand")})

#         genes_procesados.add(gen)

#     if gen in no_encontrados:
#         no_encontrados.remove(gen)
        
# df_coords = pd.DataFrame(coords_genes)

In [128]:
# mg = mygene.MyGeneInfo()
# a = mg.querymany(['440311', '100133091'], scopes = "entrezgene", fields = "symbol,genomic_pos_hg19", species = "human")

In [129]:
# nuevos_symbols = [a[i]["symbol"] for i, gen in enumerate(a)]

In [130]:
# nuevos_symbols

In [131]:
# b = mg.querymany(['LINC03009'], scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [132]:
# b

In [133]:
# no_encontrados

In [57]:
def extrae_coords_inicio_fin_0(lista_unicos):

    mg = mygene.MyGeneInfo()
    
    resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

    coords_genes = []
    no_encontrados = []

    genes_procesados = set()
    
    for resultado in resultados_coords:
        
        gen = resultado.get("query")

        if gen in genes_procesados:
            continue
    
        if resultado.get("notfound"):
            no_encontrados.append(gen)
            genes_procesados.add(gen)
            continue
    
        posicion = resultado.get("genomic_pos_hg19")
        
        if isinstance(posicion, list):
            posicion = posicion[0]
    
        if posicion:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion.get("chr")), "inicio": posicion.get("start"), "fin": posicion.get("end"), "cadena": posicion.get("strand")})

    df_coords = pd.DataFrame(coords_genes)

    return df_coords, no_encontrados

In [58]:
df_coords, no_encontrados = extrae_coords_inicio_fin(lista_unicos)

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
48 input query terms found no hit:	['SLC2A15', 'NONE', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 


In [10]:
# duplicados = df_coords["gene_symbol"].duplicated().sum()

# print(duplicados)

In [11]:
# dicc = {}
# for gen in lista_unicos:
#     dicc.update({gen: {'Chr':0, 'Inicio':0, 'SNPs': [], 'Fin':0, 'Cadena':0}})

In [12]:
# for i in range(len(df_SNPs)):
    
#     gen = df_SNPs.iloc[i]['gene_symbol']
    
#     if lista_symbols.count(gen) == 1:
        
#         dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
#         dicc[gen]['SNPs'] = df_SNPs.iloc[i]['SNP_position']
        
#     elif lista_symbols.count(gen) > 1:
            
#         dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
#         dicc[gen]['SNPs'].append(df_SNPs.iloc[i]['SNP_position'])
        
#     else:
#         continue

In [134]:
df_completo = pd.merge(df_SNPs, df_coords2, on = "gene_symbol", how = "left")

In [135]:
df_completo

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991.0,142767403.0,1.0
1,1,SYT11,155839054,C,T,1,155829300.0,155854990.0,1.0
2,12,SLC2A13,40428561,G,T,12,40148823.0,40499891.0,-1.0
3,12,SLC2A13,40478652,G,T,12,40148823.0,40499891.0,-1.0
4,12,SLC2A13,40474147,C,T,12,40148823.0,40499891.0,-1.0
...,...,...,...,...,...,...,...,...,...
1282,17,DNAH17,76425480,A,T,17,76419778.0,76573476.0,-1.0
1283,18,ASXL3,31304318,T,G,18,31158579.0,31331156.0,1.0
1284,18,MEX3C,48683589,T,G,18,48700920.0,48744674.0,-1.0
1285,20,CRLS1,6006041,T,C,20,5986736.0,6020699.0,1.0


In [136]:
for i in range(len(df_completo)):
    if '-' in df_completo.iloc[i]["SNP_position"]:
        print(i, df_completo.iloc[i]["SNP_position"])
    else:
        pass

184 42131818-41149582


In [32]:
# for i in range(len(df_completo)):
#     print(i, (df_completo.iloc[i]["Chr"], df_completo.iloc[i]["chr"]))

In [137]:
# genes_con_nan = df_completo[df_completo['chr'].isna()]['gene_symbol'].unique()
# print(f"Hay {len(genes_con_nan)} genes sin coordenadas encontradas.")
# print(genes_con_nan)

In [138]:
df_discordantes = df_completo[(df_completo['Chr'] != df_completo['chr']) & (df_completo['chr'] != 'nan') & (df_completo['chr'] != '<NA>')]
columnas = ['gene_symbol', 'Chr', 'chr', 'SNP_position', 'effect_allele', 'alternate_allele']

df_discordantes[columnas]

,gene_symbol,Chr,chr,SNP_position,effect_allele,alternate_allele
5,SLC2A15,14,NaN,40465942,T,C
6,LINC02471,12,NaN,40580440,G,A
12,LINC02210-CRHR1,17,NaN,43767773,T,C
15,LINC02210-CRHR1,17,NaN,43728376,G,A
21,LINC02210-CRHR1,17,NaN,43728376,G,A
...,...,...,...,...,...,...
1215,LINC02188,16,NaN,86883820,C,T
1218,LOC100288911,2,NaN,36287452,C,T
1226,LOC646218,10,NaN,35237731,C,T
1247,LOC108783654,17,NaN,40698158,T,C


In [14]:
df_discordantes = df_completo[(df_completo['Chr'] != df_completo['chr']) & (df_completo['chr'] != 'nan') & (df_completo['chr'] != '<NA>')]
columnas = ['gene_symbol', 'Chr', 'chr', 'SNP_position', 'effect_allele', 'alternate_allele']

df_discordantes[columnas]

,gene_symbol,Chr,chr,SNP_position,effect_allele,alternate_allele
5,SLC2A15,14,NaN,40465942,T,C
6,LINC02471,12,NaN,40580440,G,A
12,MAPT,18,17,44081064,A,G
13,LINC02210-CRHR1,17,NaN,43767773,T,C
16,LINC02210-CRHR1,17,NaN,43728376,G,A
...,...,...,...,...,...,...
1344,TAP2,6,22,32797809,C,T
1364,LOC108783654,17,NaN,40698158,T,C
1375,PAM,5,13,102365794,C,G
1381,RPS12,6,19,133210361,T,C


In [143]:
df_completo_limpio = df_completo[df_completo["Chr"] == df_completo["chr"]].reset_index(drop = True)

In [144]:
df_completo_limpio

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991.0,142767403.0,1.0
1,1,SYT11,155839054,C,T,1,155829300.0,155854990.0,1.0
2,12,SLC2A13,40428561,G,T,12,40148823.0,40499891.0,-1.0
3,12,SLC2A13,40478652,G,T,12,40148823.0,40499891.0,-1.0
4,12,SLC2A13,40474147,C,T,12,40148823.0,40499891.0,-1.0
...,...,...,...,...,...,...,...,...,...
1156,17,DNAH17,76425480,A,T,17,76419778.0,76573476.0,-1.0
1157,18,ASXL3,31304318,T,G,18,31158579.0,31331156.0,1.0
1158,18,MEX3C,48683589,T,G,18,48700920.0,48744674.0,-1.0
1159,20,CRLS1,6006041,T,C,20,5986736.0,6020699.0,1.0


In [145]:
df_completo_limpio = df_completo_limpio[~df_completo_limpio['SNP_position'].str.contains('-', na = False)].copy()

In [146]:
df_completo_limpio = df_completo_limpio.reset_index(drop = True)

In [147]:
df_completo_limpio

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991.0,142767403.0,1.0
1,1,SYT11,155839054,C,T,1,155829300.0,155854990.0,1.0
2,12,SLC2A13,40428561,G,T,12,40148823.0,40499891.0,-1.0
3,12,SLC2A13,40478652,G,T,12,40148823.0,40499891.0,-1.0
4,12,SLC2A13,40474147,C,T,12,40148823.0,40499891.0,-1.0
...,...,...,...,...,...,...,...,...,...
1155,17,DNAH17,76425480,A,T,17,76419778.0,76573476.0,-1.0
1156,18,ASXL3,31304318,T,G,18,31158579.0,31331156.0,1.0
1157,18,MEX3C,48683589,T,G,18,48700920.0,48744674.0,-1.0
1158,20,CRLS1,6006041,T,C,20,5986736.0,6020699.0,1.0


In [148]:
df_completo_limpio["SNP_position"] = df_completo_limpio["SNP_position"].astype(int)

In [149]:
df_completo_limpio.dtypes

Chr                  object
gene_symbol          object
SNP_position          int32
effect_allele        object
alternate_allele     object
chr                  object
inicio              float64
fin                 float64
cadena              float64
dtype: object

In [150]:
df_final = df_completo_limpio[(df_completo_limpio["SNP_position"] >= df_completo_limpio["inicio"]) & (df_completo_limpio["SNP_position"] <= df_completo_limpio["fin"])].copy()

In [151]:
df_final = df_final.reset_index(drop = True)

In [152]:
df_final

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991.0,142767403.0,1.0
1,1,SYT11,155839054,C,T,1,155829300.0,155854990.0,1.0
2,12,SLC2A13,40428561,G,T,12,40148823.0,40499891.0,-1.0
3,12,SLC2A13,40478652,G,T,12,40148823.0,40499891.0,-1.0
4,12,SLC2A13,40474147,C,T,12,40148823.0,40499891.0,-1.0
...,...,...,...,...,...,...,...,...,...
592,17,BRIP1,59917366,T,C,17,59758627.0,59940882.0,-1.0
593,17,DNAH17,76425480,A,T,17,76419778.0,76573476.0,-1.0
594,18,ASXL3,31304318,T,G,18,31158579.0,31331156.0,1.0
595,20,CRLS1,6006041,T,C,20,5986736.0,6020699.0,1.0


In [153]:
df_final["inicio"] = df_final["inicio"].astype(int)
df_final["fin"] = df_final["fin"].astype(int)

In [154]:
df_final = df_final.drop('chr', axis = 1)

In [155]:
df_final = df_final[['Chr', 'gene_symbol', 'inicio', 'SNP_position', 'fin', 'effect_allele', 'alternate_allele', 'cadena']]

In [156]:
df_final

,Chr,gene_symbol,inicio,SNP_position,fin,effect_allele,alternate_allele,cadena
0,6,GPR126,142622991,142758601,142767403,T,G,1.0
1,1,SYT11,155829300,155839054,155854990,C,T,1.0
2,12,SLC2A13,40148823,40428561,40499891,G,T,-1.0
3,12,SLC2A13,40148823,40478652,40499891,G,T,-1.0
4,12,SLC2A13,40148823,40474147,40499891,C,T,-1.0
...,...,...,...,...,...,...,...,...
592,17,BRIP1,59758627,59917366,59940882,T,C,-1.0
593,17,DNAH17,76419778,76425480,76573476,A,T,-1.0
594,18,ASXL3,31158579,31304318,31331156,T,G,1.0
595,20,CRLS1,5986736,6006041,6020699,T,C,1.0


In [28]:
df_final.to_csv('datosGene4PD/coordenadas_genes.csv', index = False)

In [54]:
# df_completo = df_completo.astype({'SNP_position': 'int32', 'inicio': 'int32', 'fin': 'int32'})

In [33]:
# for i in range(len(df_completo)):
#     print(df_completo.iloc[i]["inicio"] <= df_completo.iloc[i]["SNP_position"] and df_completo.iloc[i]["SNP_position"] <= df_completo.iloc[i]["fin"])

In [12]:
# for gen in lista_symbols:
#     print(gen)

In [89]:
# mg = mygene.MyGeneInfo()

In [77]:
# gen = ["GPR126"]
# resultado_prueba = mg.querymany(gen, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [78]:
# print(resultado_prueba)

In [87]:
# lista_unicos = []
# for symbol in lista_symbols:
#     if symbol not in lista_unicos:
#         lista_unicos.append(symbol)

In [90]:
# resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [24]:
# print(resultados_coords)

In [11]:
# coords_genes = []
# no_encontrados = []

# for resultado in resultados_coords:
    
#     gen = resultado.get("query")

#     if resultado.get("notfound"):
#         no_encontrados.append(gen)
#         continue

#     posicion = resultado.get("genomic_pos_hg19")
    
#     if isinstance(posicion, list):
#         posicion = posicion[0]

#     if posicion:
#         coords_genes.append({"gene_symbol": gen, "chr": str(posicion.get("chr")), "inicio": posicion.get("start"), "fin": posicion.get("end"), "cadena": posicion.get("strand")})



In [36]:
# no_encontrados

In [38]:
# coords_genes

In [12]:
# df_coords = pd.DataFrame(coords_genes)

In [91]:
# df_coords

In [79]:
# df['Unnamed: 10'].isna().all()

In [92]:
# df_corregido

In [80]:
# df_SNPs = df_corregido[["Chr", "gene_symbol", "SNP_position", "effect_allele", "alternate_allele"]]

In [81]:
# df_SNPs

In [82]:
# df_SNPs = df_SNPs[~df_SNPs['Chr'].str.contains('rs', na = False)]

In [83]:
# df_SNPs

In [19]:
# for i in range(len(df_SNPs)):
#     if 
#     loc_genes = {df_SNPs.iloc[i]['gene_symbol']: {'Chr': df_SNPs.iloc[i]['Chr'], 'SNP_pos': df_SNPs.iloc[i]['SNP_position']}}
#     break

In [20]:
# loc_genes

In [27]:
# dicc = {}
# for gen in lista_unicos:
#     dicc.update({gen: {'Chr':0, 'Inicio':0, 'SNPs': [], 'Fin':0, 'Cadena':0}})

In [28]:
# for i in range(len(df_SNPs)):
    
#     gen = df_SNPs.iloc[i]['gene_symbol']
    
#     if lista_symbols.count(gen) == 1:
        
#         dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
#         dicc[gen]['SNPs'] = df_SNPs.iloc[i]['SNP_position']
        
#     elif lista_symbols.count(gen) > 1:
            
#         dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
#         dicc[gen]['SNPs'].append(df_SNPs.iloc[i]['SNP_position'])
        
#     else:
#         continue

In [93]:
# df_corregido["gene_symbol"] = df_corregido["gene_symbol"].str.split(r'\s*[;,]\s*')

In [94]:
# df_corregido.head(20)

In [95]:
# df_corregido = df_corregido.explode("gene_symbol").reset_index(drop = True)

In [96]:
# df_corregido

In [97]:
# df_corregido.head(20)